This notebook is for reading and processing the data needed for deterministic and probabilistic nowcasting notebooks in this folder.

In [ ]:
import os

# On Colab, mount Google Drive and switch to the notebook folder.
# Locally there is nothing to mount: Jupyter already runs in this folder.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # don't attempt to remount if the drive is already mounted
    if not os.path.exists("/content/mnt/MyDrive"):
        drive.mount("mnt")
    %cd '/content/mnt/MyDrive/Colab Notebooks/ERAD-nowcasting-course-2026/notebooks/exercise_notebooks/'

# run the previous notebook to configure the environment
%run helper_input_data.ipynb

## Pre-processing steps
The pre-processing steps for the nowcasting exercise resemble the steps from block 2 and 3, and consist of the following:

### Apply data transformations
The data from FMI has already been imported after running the [helper_input_data](https://github.com/pySTEPS/ERAD-nowcasting-course-2022/blob/hands-on-users/hands-on-session-users/notebooks/helper_input_data.ipynb) notebook. The precip data is called `precip` and the metadata is called `metadata`.

In [ ]:
import numpy as np

from pysteps import motion
from pysteps.utils import transformation

# The precip variable consists of multiple timesteps, we will use a subset of them
# to compute the forecast on and the rest to validate the forecast
precip_for_forecast = precip[20:24]
precip_obs = precip[24:]

# When computing the optical flow, transforming the precipitation rates (mm/h)
# to dBR via the logarithmic transform shown in the previous exercise generally
# improves the reliability of the estimation.
precip_dbr, metadata_dbr = transformation.dB_transform(
    precip_for_forecast,
    metadata,
    threshold=0.1,
    zerovalue=-15.0
)

# Handling of NaN values has been explicitly implemented in Lucas-Kanade and VET,
# but not in DARTS. For this reason, we set all non-finite values to the minimum
# value before applying the optical flow.
precip_finite = precip_dbr.copy()
precip_finite[~np.isfinite(precip_finite)] = np.nanmin(precip_dbr)

print("metadata (dBR transformed) is: ", metadata_dbr)

### Determine the motion field
Determine the motion field with the Lucas-Kanade method.

In [3]:
# Set the method
method = "LK"

# Determine the motion field (here, using the four most recent precipitation fields)
oflow = motion.get_method(method)
motion_field = oflow(precip_finite, verbose=False)